In [1]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold, learning_curve, validation_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

project_path = Path.cwd()
if project_path.name == "notebooks":
    project_path = project_path.parent
data_path = project_path / "data" / "processed"
models_path = project_path / "models"
out_path = project_path / "docs" / "overfitting"
out_path.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(data_path / "train.csv")
val_df = pd.read_csv(data_path / "validation.csv")
test_df = pd.read_csv(data_path / "test.csv")
print("Train:", len(train_df), "| Validation:", len(val_df), "| Test:", len(test_df))

# Ngjyrat e grafikëve (dy seri, të kontrolluara për daltonizëm)
C_TRAIN, C_EVAL = "#2a78d6", "#eb6834"

Train: 803 | Validation: 178 | Test: 172


In [2]:
svm_saved = joblib.load(models_path / "svm_tuned_validation.joblib")
mlp_saved = joblib.load(models_path / "mlp_tuned_validation.joblib")

rebuilt = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 1), min_df=1)),
        ("classifier", LogisticRegression(C=10, max_iter=2000, random_state=SEED)),
    ]),
    "KNN": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 1))),
        ("classifier", KNeighborsClassifier(n_neighbors=1, weights="uniform", metric="cosine")),
    ]),
    "Random Forest": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2))),
        ("classifier", RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_leaf=1,
                                              random_state=SEED, n_jobs=1)),
    ]),
}
for model in rebuilt.values():
    model.fit(train_df["text"], train_df["label"])

models = {"SVM (LinearSVC)": svm_saved, **rebuilt, "Neural Network (MLP)": mlp_saved}

# Kontrolli i riprodhueshmërisë kundrejt rezultateve të dokumentuara (Validation)
documented = pd.read_csv(models_path / "tuned_results.csv").set_index("Model")
name_map = {"SVM (LinearSVC)": "SVM", "Neural Network (MLP)": "Neural Network (MLP)"}
check = []
for name, model in models.items():
    pred = model.predict(val_df["text"])
    doc = documented.loc[name_map.get(name, name)]
    check.append({
        "Model": name,
        "Val Accuracy (tani)": round(accuracy_score(val_df["label"], pred), 4),
        "Val Accuracy (dokumentuar)": doc["Accuracy"],
        "Val Macro F1 (tani)": round(f1_score(val_df["label"], pred, average="macro"), 4),
        "Val Macro F1 (dokumentuar)": doc["Macro F1"],
    })
check = pd.DataFrame(check)
check["Përputhet"] = (check["Val Accuracy (tani)"] - check["Val Accuracy (dokumentuar)"]).abs().lt(1e-4) & \
                     (check["Val Macro F1 (tani)"] - check["Val Macro F1 (dokumentuar)"]).abs().lt(1e-4)
check.to_csv(out_path / "00_reproducibility_check.csv", index=False)
check

,Model,Val Accuracy (tani),Val Accuracy (dokumentuar),Val Macro F1 (tani),Val Macro F1 (dokumentuar),Përputhet
0,SVM (LinearSVC),0.9944,0.9944,0.9952,0.9952,True
1,Logistic Regression,0.9775,0.9775,0.9787,0.9787,True
2,KNN,0.9775,0.9775,0.9777,0.9777,True
3,Random Forest,0.9382,0.9382,0.9386,0.9386,True
4,Neural Network (MLP),0.9775,0.9775,0.9790,0.9790,True


In [3]:
rows = []
for name, model in models.items():
    row = {"Model": name}
    for split, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        pred = model.predict(df["text"])
        row[f"{split} Accuracy"] = accuracy_score(df["label"], pred)
        row[f"{split} Macro F1"] = f1_score(df["label"], pred, average="macro")
    rows.append(row)
split_results = pd.DataFrame(rows)
split_results["Gap Train-Val (Acc)"] = split_results["Train Accuracy"] - split_results["Validation Accuracy"]
split_results["Gap Train-Test (Acc)"] = split_results["Train Accuracy"] - split_results["Test Accuracy"]
split_results.to_csv(out_path / "01_train_val_test.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(split_results))
for offset, split, color, marker in [(-0.18, "Train", C_TRAIN, "o"), (0.0, "Validation", "#6b7280", "s"), (0.18, "Test", C_EVAL, "D")]:
    ax.scatter(split_results[f"{split} Accuracy"] * 100, y + offset, label=split, color=color, marker=marker, s=60, zorder=3)
ax.set_yticks(y, split_results["Model"])
ax.invert_yaxis()
ax.set_xlabel("Accuracy (%)")
ax.set_xlim(88, 100.5)
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower left")
ax.set_title("Train, Validation dhe Test Accuracy për çdo model")
fig.tight_layout()
fig.savefig(out_path / "01_train_val_test.png", dpi=150)
(split_results.set_index("Model") * 100).round(2)

,Train Accuracy,Train Macro F1,Validation Accuracy,Validation Macro F1,Test Accuracy,Test Macro F1,Gap Train-Val (Acc),Gap Train-Test (Acc)
Model,,,,,,,,
SVM (LinearSVC),100.0,100.0,99.44,99.52,98.84,98.52,0.56,1.16
Logistic Regression,100.0,100.0,97.75,97.87,98.84,98.52,2.25,1.16
KNN,100.0,100.0,97.75,97.77,97.09,97.26,2.25,2.91
Random Forest,100.0,100.0,93.82,93.86,93.60,92.82,6.18,6.40
Neural Network (MLP),100.0,100.0,97.75,97.90,98.84,98.49,2.25,1.16


In [4]:
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
sizes = np.linspace(0.1, 1.0, 10)
lc_rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, name in zip(axes, ["SVM (LinearSVC)", "Logistic Regression"]):
    n, tr_s, cv_s = learning_curve(clone(models[name]), train_df["text"], train_df["label"],
                                   groups=train_df["group_id"], cv=cv5, train_sizes=sizes,
                                   scoring="f1_macro", shuffle=True, random_state=SEED, n_jobs=1)
    for k in range(len(n)):
        lc_rows.append({"Model": name, "Train size": int(n[k]),
                        "Train Macro F1 mean": tr_s[k].mean(), "CV Macro F1 mean": cv_s[k].mean(),
                        "CV Macro F1 std": cv_s[k].std()})
    ax.plot(n, tr_s.mean(1) * 100, "o-", color=C_TRAIN, label="Train")
    ax.plot(n, cv_s.mean(1) * 100, "D-", color=C_EVAL, label="CV (brenda Train)")
    ax.fill_between(n, (cv_s.mean(1) - cv_s.std(1)) * 100, (cv_s.mean(1) + cv_s.std(1)) * 100, color=C_EVAL, alpha=0.15)
    ax.set_title(name)
    ax.set_xlabel("Numri i teksteve të trajnimit")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Macro F1 (%)")
axes[0].legend(loc="lower right")
fig.suptitle("Learning Curve (5-fold StratifiedGroupKFold brenda Train)")
fig.tight_layout()
fig.savefig(out_path / "02_learning_curve.png", dpi=150)
learning = pd.DataFrame(lc_rows)
learning.to_csv(out_path / "02_learning_curve.csv", index=False)
learning.round(4)

,Model,Train size,Train Macro F1 mean,CV Macro F1 mean,CV Macro F1 std
0,SVM (LinearSVC),64,1.0,0.6061,0.0612
1,SVM (LinearSVC),128,1.0,0.7739,0.0283
2,SVM (LinearSVC),192,1.0,0.8319,0.0275
3,SVM (LinearSVC),256,1.0,0.8718,0.0116
4,SVM (LinearSVC),321,1.0,0.9037,0.0206
5,SVM (LinearSVC),385,1.0,0.9275,0.0147
6,SVM (LinearSVC),449,1.0,0.9398,0.0139
7,SVM (LinearSVC),513,1.0,0.9549,0.0099
8,SVM (LinearSVC),577,1.0,0.9626,0.0147
9,SVM (LinearSVC),642,1.0,0.9763,0.0034


In [5]:
C_values = np.logspace(-3, 3, 13)
tr_s, cv_s = validation_curve(clone(svm_saved), train_df["text"], train_df["label"], groups=train_df["group_id"],
                              param_name="classifier__C", param_range=C_values, cv=cv5, scoring="f1_macro", n_jobs=1)
val_scores = []
for C in C_values:
    m = clone(svm_saved).set_params(classifier__C=C).fit(train_df["text"], train_df["label"])
    val_scores.append(f1_score(val_df["label"], m.predict(val_df["text"]), average="macro"))
reg = pd.DataFrame({"C": C_values, "Train Macro F1": tr_s.mean(1), "CV Macro F1 mean": cv_s.mean(1),
                    "CV Macro F1 std": cv_s.std(1), "Validation Macro F1": val_scores})
reg.to_csv(out_path / "03_regularization_curve.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.3))
ax.semilogx(C_values, reg["Train Macro F1"] * 100, "o-", color=C_TRAIN, label="Train")
ax.semilogx(C_values, reg["CV Macro F1 mean"] * 100, "D-", color=C_EVAL, label="CV (brenda Train)")
ax.fill_between(C_values, (reg["CV Macro F1 mean"] - reg["CV Macro F1 std"]) * 100,
                (reg["CV Macro F1 mean"] + reg["CV Macro F1 std"]) * 100, color=C_EVAL, alpha=0.15)
ax.semilogx(C_values, reg["Validation Macro F1"] * 100, "s--", color="#6b7280", label="Validation")
ax.axvline(10, color="#111827", linestyle=":", linewidth=1)
ax.annotate("C = 10 (i zgjedhur)", xy=(10, 0.0), xycoords=("data", "axes fraction"),
            xytext=(4, 6), textcoords="offset points", ha="left", fontsize=9)
ax.set_xlabel("C (shkallë logaritmike)")
ax.set_ylabel("Macro F1 (%)")
ax.set_title("Regularization Curve për LinearSVC")
ax.grid(alpha=0.3)
ax.legend(loc="lower right", bbox_to_anchor=(1.0, 0.12))
fig.tight_layout()
fig.savefig(out_path / "03_regularization_curve.png", dpi=150)
reg.round(4)

,C,Train Macro F1,CV Macro F1 mean,CV Macro F1 std,Validation Macro F1
0,0.0010,0.8617,0.7318,0.0262,0.7646
1,0.0032,0.9618,0.8698,0.0261,0.8969
2,0.0100,0.9860,0.9024,0.0141,0.9317
3,0.0316,0.9954,0.9245,0.0107,0.9480
4,0.1000,0.9994,0.9467,0.0107,0.9595
5,0.3162,1.0000,0.9615,0.0040,0.9664
6,1.0000,1.0000,0.9679,0.0053,0.9900
7,3.1623,1.0000,0.9736,0.0032,0.9900
8,10.0000,1.0000,0.9763,0.0034,0.9952
9,31.6228,1.0000,0.9763,0.0034,0.9952


In [6]:
pool = pd.concat([train_df, val_df], ignore_index=True)
cv_rows = []
cv_pairs = []
for repeat in range(5):
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
    for fold, (tr_idx, te_idx) in enumerate(splitter.split(pool["text"], pool["label"], pool["group_id"])):
        Xtr, ytr = pool["text"].iloc[tr_idx], pool["label"].iloc[tr_idx]
        Xte, yte = pool["text"].iloc[te_idx], pool["label"].iloc[te_idx]
        for name, model in models.items():
            m = clone(model).fit(Xtr, ytr)
            pred_te, pred_tr = m.predict(Xte), m.predict(Xtr)
            cv_rows.append({"Model": name, "Repeat": repeat, "Fold": fold,
                            "Train Accuracy": accuracy_score(ytr, pred_tr),
                            "Held-out Accuracy": accuracy_score(yte, pred_te),
                            "Held-out Macro F1": f1_score(yte, pred_te, average="macro")})
            if name == "SVM (LinearSVC)":
                cv_pairs += [(a, b) for a, b in zip(yte, pred_te) if a != b]
cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(out_path / "04_group_cv_folds.csv", index=False)
cv_summary = cv_df.groupby("Model").agg(
    train_acc_mean=("Train Accuracy", "mean"),
    heldout_acc_mean=("Held-out Accuracy", "mean"), heldout_acc_std=("Held-out Accuracy", "std"),
    heldout_acc_min=("Held-out Accuracy", "min"), heldout_acc_max=("Held-out Accuracy", "max"),
    heldout_f1_mean=("Held-out Macro F1", "mean"), heldout_f1_std=("Held-out Macro F1", "std"),
).reindex(list(models))
cv_summary["gap_train_heldout"] = cv_summary["train_acc_mean"] - cv_summary["heldout_acc_mean"]
cv_summary.to_csv(out_path / "04_group_cv_summary.csv")

fig, ax = plt.subplots(figsize=(9, 4.3))
data = [cv_df.loc[cv_df["Model"] == m, "Held-out Accuracy"] * 100 for m in models]
ax.boxplot(data, vert=False, tick_labels=list(models), widths=0.5)
ax.invert_yaxis()
ax.set_xlabel("Accuracy në fold-in e mbajtur jashtë (%)")
ax.set_title("Luhatja e Accuracy: 5×5 StratifiedGroupKFold mbi Train+Validation")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(out_path / "04_group_cv_variance.png", dpi=150)
(cv_summary * 100).round(2)

,train_acc_mean,heldout_acc_mean,heldout_acc_std,heldout_acc_min,heldout_acc_max,heldout_f1_mean,heldout_f1_std,gap_train_heldout
Model,,,,,,,,
SVM (LinearSVC),100.0,98.04,0.88,95.41,98.99,98.09,0.87,1.96
Logistic Regression,100.0,97.72,1.00,94.90,98.99,97.74,0.99,2.28
KNN,100.0,96.84,1.51,93.85,100.00,96.89,1.46,3.16
Random Forest,100.0,94.37,1.62,89.80,97.46,94.28,1.75,5.63
Neural Network (MLP),100.0,97.09,1.20,93.88,98.97,97.14,1.17,2.91


In [7]:
y_test = test_df["label"].to_numpy()
pred_test = svm_saved.predict(test_df["text"])
correct, n_test = int((pred_test == y_test).sum()), len(y_test)

def wilson(k, n, conf=0.95):
    z = norm.ppf(1 - (1 - conf) / 2)
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half

rng = np.random.default_rng(SEED)
boot_acc, boot_f1 = [], []
for _ in range(2000):
    idx = rng.integers(0, n_test, n_test)
    boot_acc.append(accuracy_score(y_test[idx], pred_test[idx]))
    boot_f1.append(f1_score(y_test[idx], pred_test[idx], average="macro"))
w_lo, w_hi = wilson(correct, n_test)
uncertainty = pd.DataFrame([
    {"Metrika": "Accuracy", "Vlera": correct / n_test, "Metoda": "Wilson 95%", "Kufiri i poshtëm": w_lo, "Kufiri i sipërm": w_hi},
    {"Metrika": "Accuracy", "Vlera": correct / n_test, "Metoda": "Bootstrap 95%", "Kufiri i poshtëm": np.percentile(boot_acc, 2.5), "Kufiri i sipërm": np.percentile(boot_acc, 97.5)},
    {"Metrika": "Macro F1", "Vlera": f1_score(y_test, pred_test, average="macro"), "Metoda": "Bootstrap 95%", "Kufiri i poshtëm": np.percentile(boot_f1, 2.5), "Kufiri i sipërm": np.percentile(boot_f1, 97.5)},
])
uncertainty.to_csv(out_path / "05_test_uncertainty.csv", index=False)
print(f"Saktë: {correct}/{n_test}")
uncertainty.round(4)

Saktë: 170/172


,Metrika,Vlera,Metoda,Kufiri i poshtëm,Kufiri i sipërm
0,Accuracy,0.9884,Wilson 95%,0.9586,0.9968
1,Accuracy,0.9884,Bootstrap 95%,0.9709,1.0000
2,Macro F1,0.9852,Bootstrap 95%,0.9533,1.0000


In [8]:
sim_vec = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2))
sim_vec.fit(pd.concat([train_df, val_df, test_df])["text"])
X_train_sim = sim_vec.transform(train_df["text"])
bins = [(0.0, 0.5), (0.5, 0.7), (0.7, 0.85)]
sim_rows, hist_data = [], {}
for split, df in [("Validation", val_df), ("Test", test_df)]:
    sim = cosine_similarity(sim_vec.transform(df["text"]), X_train_sim)
    best, nearest = sim.max(1), sim.argmax(1)
    hist_data[split] = best
    ok = svm_saved.predict(df["text"]) == df["label"].to_numpy()
    nn_ok = train_df["label"].to_numpy()[nearest] == df["label"].to_numpy()
    for lo, hi in bins:
        m = (best >= lo) & (best < hi)
        sim_rows.append({"Split": split, "Similarity bin": f"{lo:.2f}-{hi:.2f}", "N": int(m.sum()),
                         "Share": m.mean(), "SVM Accuracy": ok[m].mean() if m.any() else np.nan,
                         "Nearest-neighbour Accuracy": nn_ok[m].mean() if m.any() else np.nan})
    sim_rows.append({"Split": split, "Similarity bin": "all", "N": len(df), "Share": 1.0,
                     "SVM Accuracy": ok.mean(), "Nearest-neighbour Accuracy": nn_ok.mean()})
    strict = best < 0.7
    sim_rows.append({"Split": split, "Similarity bin": "strict (<0.70)", "N": int(strict.sum()),
                     "Share": strict.mean(), "SVM Accuracy": ok[strict].mean(),
                     "Nearest-neighbour Accuracy": nn_ok[strict].mean()})
similarity = pd.DataFrame(sim_rows)
similarity.to_csv(out_path / "06_similarity_analysis.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(hist_data["Validation"], bins=20, range=(0, 1), alpha=0.6, color="#6b7280", label="Validation")
ax.hist(hist_data["Test"], bins=20, range=(0, 1), alpha=0.6, color=C_EVAL, label="Test")
ax.axvline(0.85, color="#111827", linestyle=":", linewidth=1)
ax.text(0.855, ax.get_ylim()[1] * 0.9, "pragu i grupimit 0.85", fontsize=9)
ax.set_xlabel("Ngjashmëria me tekstin më të afërt të Train (cosine)")
ax.set_ylabel("Numri i teksteve")
ax.set_title("Sa të ngjashme janë tekstet e vlerësimit me Train")
ax.legend()
fig.tight_layout()
fig.savefig(out_path / "06_similarity_hist.png", dpi=150)
similarity.round(4)

,Split,Similarity bin,N,Share,SVM Accuracy,Nearest-neighbour Accuracy
0,Validation,0.00-0.50,126,0.7079,0.9921,0.9365
1,Validation,0.50-0.70,48,0.2697,1.0000,1.0000
2,Validation,0.70-0.85,4,0.0225,1.0000,1.0000
3,Validation,all,178,1.0000,0.9944,0.9551
4,Validation,strict (<0.70),174,0.9775,0.9943,0.9540
5,Test,0.00-0.50,123,0.7151,0.9837,0.9187
6,Test,0.50-0.70,24,0.1395,1.0000,1.0000
7,Test,0.70-0.85,25,0.1453,1.0000,1.0000
8,Test,all,172,1.0000,0.9884,0.9419
9,Test,strict (<0.70),147,0.8547,0.9864,0.9320


In [9]:
tfidf, clf = svm_saved.named_steps["tfidf"], svm_saved.named_steps["classifier"]
features = tfidf.get_feature_names_out()
classes = list(clf.classes_)

def top_words(text, label, k=4):
    vec = tfidf.transform([text]).toarray()[0]
    contrib = vec * clf.coef_[classes.index(label)]
    idx = np.argsort(contrib)[::-1][:k]
    return ", ".join(f"{features[i]} ({contrib[i]:+.2f})" for i in idx if vec[i] > 0)

err_rows = []
for split, df in [("Validation", val_df), ("Test", test_df)]:
    pred = svm_saved.predict(df["text"])
    for text, true, p in zip(df["text"], df["label"], pred):
        if true != p:
            err_rows.append({"Split": split, "E vërteta": true, "E parashikuara": p,
                             "Fjalët drejt klasës së parashikuar": top_words(text, p),
                             "Fjalët drejt klasës së vërtetë": top_words(text, true), "Teksti": text})
errors = pd.DataFrame(err_rows)
errors.to_csv(out_path / "07_errors_val_test.csv", index=False)

pairs = pd.Series([f"{a} → {b}" for a, b in cv_pairs]).value_counts().head(10).rename_axis("E vërteta → E parashikuara").reset_index(name="Herë në CV")
pairs.to_csv(out_path / "07_cv_confusion_pairs.csv", index=False)
display(errors.drop(columns="Teksti"))
pairs

,Split,E vërteta,E parashikuara,Fjalët drejt klasës së parashikuar,Fjalët drejt klasës së vërtetë
0,Validation,Chicken pox,Common Cold,"can (+0.25), barely (+0.22), unable (+0.11), k...","lethargic (+0.31), no (+0.12), can (+0.11), fe..."
1,Test,allergy,Common Cold,"can (+0.27), my (+0.21), throat (+0.20), lot (...","always (+0.19), throat (+0.17), taste (+0.14),..."
2,Test,drug reaction,diabetes,"want (+0.32), do (+0.16), have (+0.08), sense ...","sex (+0.35), brain (+0.32), fog (+0.23), for (..."


,E vërteta → E parashikuara,Herë në CV
0,drug reaction → diabetes,13
1,Dengue → Typhoid,8
2,Dengue → Chicken pox,7
3,peptic ulcer disease → gastroesophageal reflux...,7
4,diabetes → drug reaction,5
5,Chicken pox → Typhoid,5
6,diabetes → urinary tract infection,5
7,Psoriasis → peptic ulcer disease,5
8,Common Cold → allergy,5
9,drug reaction → peptic ulcer disease,4


In [10]:
svm_row = split_results.set_index("Model").loc["SVM (LinearSVC)"]
summary = {
    "svm": {
        "train_accuracy": svm_row["Train Accuracy"], "validation_accuracy": svm_row["Validation Accuracy"],
        "test_accuracy": svm_row["Test Accuracy"], "gap_train_test": svm_row["Gap Train-Test (Acc)"],
        "test_wilson_95": [w_lo, w_hi],
        "group_cv_heldout_accuracy_mean": cv_summary.loc["SVM (LinearSVC)", "heldout_acc_mean"],
        "group_cv_heldout_accuracy_std": cv_summary.loc["SVM (LinearSVC)", "heldout_acc_std"],
        "group_cv_heldout_accuracy_range": [cv_summary.loc["SVM (LinearSVC)", "heldout_acc_min"],
                                            cv_summary.loc["SVM (LinearSVC)", "heldout_acc_max"]],
        "best_C_by_cv": float(reg.loc[reg["CV Macro F1 mean"].idxmax(), "C"]),
        "test_strict_accuracy_lt_0_70": float(similarity.query("Split == 'Test' and `Similarity bin` == 'strict (<0.70)'")["SVM Accuracy"].iloc[0]),
        "test_nearest_neighbour_accuracy": float(similarity.query("Split == 'Test' and `Similarity bin` == 'all'")["Nearest-neighbour Accuracy"].iloc[0]),
    },
    "reproducibility_all_match": bool(check["Përputhet"].all()),
}
(out_path / "overfitting_summary.json").write_text(json.dumps(summary, indent=2, default=float), encoding="utf-8")
summary

{'svm': {'train_accuracy': np.float64(1.0),
  'validation_accuracy': np.float64(0.9943820224719101),
  'test_accuracy': np.float64(0.9883720930232558),
  'gap_train_test': np.float64(0.011627906976744207),
  'test_wilson_95': [np.float64(0.958600691496456),
   np.float64(0.9968053949106748)],
  'group_cv_heldout_accuracy_mean': np.float64(0.9804308272502616),
  'group_cv_heldout_accuracy_std': np.float64(0.008771771692596686),
  'group_cv_heldout_accuracy_range': [np.float64(0.9540816326530612),
   np.float64(0.98989898989899)],
  'best_C_by_cv': 10.0,
  'test_strict_accuracy_lt_0_70': 0.9863945578231292,
  'test_nearest_neighbour_accuracy': 0.9418604651162791},
 'reproducibility_all_match': True}